### Great — Phase 4 is the AI core. We’ll train a reproducible forecasting model, track everything with MLflow, evaluate it, and persist predictions back into the Gold layer. I’ll give you copy-paste-ready notebook cells (PySpark + Spark ML + MLflow) and a clear checklist so a beginner can follow.
###  
### - Decision choices (kept simple & robust for Databricks Free):
###  
### - Predict next-day sales per (store_id, product_family) (lead-1). This is actionable for replenishment and keeps the problem simple and explainable.
### 
### - Use features created in Phase 3 (avg_7d_sales, avg_14d_sales, sales_trend, promo_ratio, onpromotion_count) plus categorical encodings for store_id and product_family.
### 
### - Use Spark ML RandomForestRegressor (no external libs required) and log with MLflow (Databricks has MLflow built in).

In [0]:
spark.sql("USE inventory_ai")

from pyspark.sql.functions import col, lead, lag, avg, sqrt, abs as _abs
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
import mlflow
import mlflow.spark


In [0]:
# load gold features
features_df = spark.table("gold_demand_features")

# create label = next day sales per store+product
w = Window.partitionBy("store_id", "product_family").orderBy("date")

labeled_df = (
    features_df
    .withColumn("label", lead(col("daily_sales"), 1).over(w))
    # keep the row that has a label (i.e. we can predict the next day)
    .filter(col("label").isNotNull())
)

# Optional: persist intermediate training table
labeled_df.write.format("delta").mode("overwrite").saveAsTable("inventory_ai.gold_train_features")

display(labeled_df.limit(5))


In [0]:
# find max date
max_date = labeled_df.agg({"date": "max"}).collect()[0][0]
print(f"Max date in dataset: {max_date}")

# choose test_cutoff_date = max_date - 28 days
from pyspark.sql.functions import date_sub
# Fix: alias the date_sub result as 'date' for aggregation
test_cutoff = labeled_df.select(date_sub(col("date"), 28).alias("date")).agg({"date": "max"}).collect()[0][0]

# simpler approach: compute distinct dates, pick cutoff by rank
dates = labeled_df.select("date").distinct().orderBy(col("date").desc())
date_list = [row.date for row in dates.limit(1000).collect()]
# if less than 60 days, choose small holdout; fallback 28
if len(date_list) >= 29:
    cutoff = date_list[28]  # last 28 days as test
else:
    cutoff = date_list[-1]  # minimal fallback

print(f"Cutoff date for train/test split: {cutoff}")

train_df = labeled_df.filter(col("date") < cutoff)
test_df  = labeled_df.filter(col("date") >= cutoff)

print("Train rows:", train_df.count(), "Test rows:", test_df.count())

In [0]:
# Define categorical columns
cat_cols = ["store_id", "product_family"]

# StringIndexer for each category
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]

# OneHotEncoder (sparse vector)
ohe = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols],
                    outputCols=[f"{c}_ohe" for c in cat_cols],
                    handleInvalid="keep")

# numeric features
numeric_cols = ["daily_sales", "avg_7d_sales", "avg_14d_sales",
                "sales_trend", "promo_ratio", "onpromotion_count"]

assembler_inputs = [f"{c}_ohe" for c in cat_cols] + numeric_cols

assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")


In [0]:
rf = RandomForestRegressor(featuresCol="features", labelCol="label", predictionCol="prediction", numTrees=50, maxDepth=8)

pipeline = Pipeline(stages=indexers + [ohe, assembler, rf])


In [0]:
# Start MLflow run
with mlflow.start_run(run_name="rf_next_day_sales"):
    # log some params
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("num_trees", 50)
    mlflow.log_param("max_depth", 8)

    # fit pipeline on train set
    model = pipeline.fit(train_df)

    # Predict on test set
    preds = model.transform(test_df).select("date","store_id","product_family","label","prediction")
    
    # Compute evaluation metrics (RMSE, MAE) in Spark
    eval_df = preds.withColumn("sq_err", (col("prediction") - col("label"))**2)\
                   .withColumn("abs_err", _abs(col("prediction") - col("label")))
    
    metrics = eval_df.agg({"sq_err": "avg", "abs_err": "avg"}).collect()[0].asDict()
    mse = metrics["avg(sq_err)"]
    mae = metrics["avg(abs_err)"]
    rmse = float(mse**0.5) if mse is not None else None

    # log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", float(mae))

    # log the Spark ML pipeline model
    mlflow.spark.log_model(model, artifact_path="rf_next_day_sales_model", dfs_tmpdir="/Volumes/cust_churn/default/ml")
    
    run_id = mlflow.active_run().info.run_id
    print(f"MLflow run id: {run_id}, RMSE: {rmse}, MAE: {mae}")


In [0]:
# join predictions back with features for context (use model.transform on full features set for scoring)
full_pred_df = model.transform(labeled_df) \
    .select("date", "store_id", "product_family", "daily_sales", "onpromotion_count", "avg_7d_sales", "avg_14d_sales", "sales_trend", "promo_ratio", "prediction")

# Save as gold forecast table
full_pred_df.write.format("delta").mode("overwrite").saveAsTable("inventory_ai.gold_demand_forecast")

spark.sql("SELECT * FROM inventory_ai.gold_demand_forecast ORDER BY date DESC LIMIT 10").show()


In [0]:
# sample comparisons
spark.sql("""
SELECT store_id, product_family, date, daily_sales AS actual, prediction
FROM inventory_ai.gold_demand_forecast
ORDER BY date DESC
LIMIT 25
""").show(truncate=False)

# overall metrics from the persisted preds (validation)
preds_df = spark.table("inventory_ai.gold_demand_forecast")\
    .withColumn("label_nextday", lead(col("daily_sales"), 1).over(Window.partitionBy("store_id","product_family").orderBy("date")))\
    .filter(col("label_nextday").isNotNull())

from pyspark.sql.functions import mean as _mean
eval_metrics = preds_df.withColumn("sq_err", (col("prediction") - col("label_nextday"))**2)\
                       .withColumn("abs_err", _abs(col("prediction") - col("label_nextday")))\
                       .agg(_mean("sq_err").alias("mse"), _mean("abs_err").alias("mae")).collect()[0]
print("RMSE:", float(eval_metrics["mse"]**0.5), "MAE:", float(eval_metrics["mae"]))
